# 처방전 구조화·약품 매칭 프롬프트 실험

`app/prompts/structure.md`를 고치면서 gpt-4o 출력을 확인하는 노트북입니다. 위에서부터 순서대로 실행하세요.

- 키 없이 흐름만 보려면 2·3번 셀을 실행한 뒤 **마지막 셀(오프라인)** 로 가면 됩니다.
- 실제 처방전은 넣지 마세요. `data/samples/`의 가상 샘플만 사용합니다.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from app import config
print('모델:', config.OPENAI_MODEL)

In [ ]:
SAMPLE = 'rx_001'   # rx_001 ~ rx_003
ocr = (ROOT / 'data' / 'samples' / f'{SAMPLE}.txt').read_text(encoding='utf-8')
expected = json.loads((ROOT / 'data' / 'expected' / f'{SAMPLE}.extracted.json').read_text(encoding='utf-8'))
print(ocr)

In [ ]:
# 3. 키와 모델이 되는지 한 줄 호출로 확인 (.env에 OPENAI_API_KEY 필요)
from app.llm import get_chat_model

print(get_chat_model().invoke('한 문장으로 인사해 주세요.').content)

In [ ]:
# 4. gpt-4o로 처방전 구조화
from app.nodes.structure import structure_node
from app.nodes.match import match_node

state = {'prescription_id': 'P0001', 'ocr': ocr}
state.update(structure_node(state))
print(state['status'], state['notice'])
print(json.dumps(state.get('order'), ensure_ascii=False, indent=2))

In [ ]:
# 5. 정답(expected)과 약품별로 비교
keys = ('name', 'dose_per_time', 'times_per_day', 'days')
got = [tuple(d[k] for k in keys) for d in state['order']['drugs']]
want = [tuple(d[k] for k in keys) for d in expected['drugs']]
print('모두 일치' if got == want else '다른 부분 있음')
for g, w in zip(got, want):
    print('  ok ' if g == w else '  !! ', g, '<->', w)

In [ ]:
# 6. 약품 매칭
state.update(match_node(state))
print(state['status'], state['notice'])
for d in state['order']['drugs']:
    print(d['name'], '->', d['item_code'], d['match_method'], d['warnings'])
    for c in d['match_candidates']:
        print('    후보:', c['item_code'], c['name'], c['score'], c['reason'])

In [ ]:
# 7. (오프라인) 키 없이 흐름만 확인: gpt-4o 대신 정답 추출 결과를 사용
from app.nodes.structure import structure_node
from app.nodes.match import match_node

offline = {'prescription_id': 'P0001', 'ocr': ocr}
offline.update(structure_node(offline, extract_fn=lambda text: expected))
offline.update(match_node(offline))
print(offline['status'], offline['notice'])
for d in offline['order']['drugs']:
    print(d['name'], '->', d['item_code'], d['match_method'], d['warnings'])